# Franken training

In this notebook we will explore the training of `franken` on a small dataset of DFT calculations.

We use the H2O data obtained from DFT calculations (using RPBE+D3 theory), originally collected by [Montero de Hijes et al.](https://doi.org/10.1063/5.0197105).

To showcase the sample efficiency of our model—and to fit within a small computational budget—we fine-tune the [MACE-MP0 foundation model](https://mace-docs.readthedocs.io/en/latest/guide/foundation_models.html) with only 8 new samples.

Note that while the original zero-shot MACE model has poor accuracy on this dataset (which is out of distribution with respect to the foundation model's training data), the fine-tuned `franken` model is very accurate at predicting energy and forces!

Check the [documentation](https://franken.readthedocs.io/) for a description of each argument.

This notebook is also available on [Google Colab](https://colab.research.google.com/github/CSML-IIT-UCL/franken/blob/main/notebooks/training.ipynb) for easy running.

In [ ]:
try:
    import franken
except ImportError:
    %pip install "franken[mace]"
    import franken

In [ ]:
import json

import ase.io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from franken.autotune import autotune
from franken.backbones.utils import CacheDir
from franken.config import (
    AutotuneConfig,
    DatasetConfig,
    GaussianRFConfig,
    HPSearchConfig,
    MaceBackboneConfig,
    SolverConfig,
)
from franken.datasets.registry import DATASET_REGISTRY

## Load the data

We use the water dataset available in the dataset registry. In particular, we load 8 random samples from the training set and the full validation set, which contains 189 structures. The dataset will be downloaded into Franken's cache directory (`CacheDir.get()`).

To use your own dataset, point the `data_path` argument to a file readable by `ase.io.read`, such as an extended XYZ file. Franken expects ASE units: eV for energies, eV/Å for forces, and eV/Å³ for stress.

> **Note:** The cache directory is used to store model backbones and downloaded datasets.
> It defaults to `$HOME/.franken` for the current user and can be configured by setting the
> `FRANKEN_CACHE_DIR` environment variable before importing Franken. For example:
> ```python
> import os
> os.environ["FRANKEN_CACHE_DIR"] = "/path/to/my/cache/franken-cache"
> ```

In [ ]:
train_path = DATASET_REGISTRY.get_path("water", "train", base_path=CacheDir.get())
val_path = DATASET_REGISTRY.get_path("water", "val", base_path=CacheDir.get())

print(f"Train path: {train_path}")
print(f"Validation path: {val_path}")

In [ ]:
# The registered files are standard ASE-readable extended XYZ files.
train_atoms = ase.io.read(train_path, index=0)
val_atoms = ase.io.read(val_path, index=0)

print(f"Train atoms: {train_atoms}")
print(f"Validation atoms: {val_atoms}")

We need to configure the GNN backbone we wish to use since `franken` extracts pre-trained atomic features from it.
We will use the [**MACE MP0 small** backbone](https://github.com/ACEsuit/mace-foundations) (identified by its `mace_mp/small` ID), with features extracted at the 2nd layer.

The backbone will be downloaded as necessary into the cache directory.

In [ ]:
gnn_config = MaceBackboneConfig(
    path_or_id="mace_mp/small",
    interaction_block=2,
)

## Training a model

`franken` fits a potential by combining two components:
- a **kernel**, approximated with random features (e.g. a Gaussian kernel with a given `length_scale`);
- a **linear solver**, regularized with `l2_penalty` and weighting the energy/force/stress losses (e.g. `force_weight`).

Rather than picking one specific combination of these hyperparameters (HPs) by hand, `franken` provides an `autotune` utility that performs a grid-search over them and selects the best model according to a validation set. This mirrors what the `franken.autotune` command-line tool does, so what you learn here transfers directly to the CLI.

**Why do we run `autotune` twice below?** The two HP groups are not equally expensive to search:
- **Kernel HPs** (like `length_scale`) are expensive: for every value, the whole random feature map has to be recomputed and the model refit from scratch.
- **Solver HPs** (like `l2_penalty` and `force_weight`) are cheap: once the feature map is computed, `autotune` can efficiently re-fit and re-score many combinations by reusing it.

So we first fix a reasonable `length_scale` and search only the cheap solver HPs (**Step 1**). Then, we broaden the search to also tune `length_scale` (**Step 2**), to check whether it further improves the model.

> **Tip — avoiding the length-scale search altogether:** `franken` also provides a `MultiscaleGaussianRFConfig` kernel, which combines several length scales into a *single* model (instead of training one model per length scale). This removes the need to search `length_scale` at all, at the cost of a somewhat larger random feature map:
> ```python
> from franken.config import MultiscaleGaussianRFConfig
>
> rf_config = MultiscaleGaussianRFConfig(
>     num_random_features=512,
>     length_scale_low=4.0,
>     length_scale_high=24.0,
>     length_scale_num=4,
>     rng_seed=42,
> )
> ```
> This notebook uses `GaussianRFConfig` instead, so that the two-step `autotune` search below is easier to follow.

The configuration we build below is equivalent to running from the command line:

```bash
franken.autotune \
    --train-path $HOME/.franken/water/ML_AB_dataset_1.xyz \
    --val-path $HOME/.franken/water/ML_AB_dataset_2-val.xyz \
    --max-train-samples 8 \
    --l2-penalty="(-10, -5, 5, log)" \
    --force-weight="(-2, 2, 5, log)" \
    --metrics energy_MAE forces_MAE \
    --seed 42 \
    --jac-chunk-size auto \
    --run-dir ./results \
    --backbone=mace --mace.path-or-id mace_mp/small --mace.interaction-block 2 \
    --rf=gaussian --gaussian.num-rf 512 \
    --gaussian.length-scale="[1., 5., 10., 20., 30.]"
```

A **validation dataset** distinct from the training set is important so that model selection does not leak training data. We reuse the `train_path` and `val_path` loaded earlier for the water dataset.

> **Note — full manual control:** if you need to fully customize the training loop (e.g. custom batching, logging, or a training procedure not covered by `autotune`), `franken` also exposes the lower-level `FrankenPotential` model and `LowMemRandomFeaturesTrainer`/`RandomFeaturesTrainer` classes directly. This is not covered here since it has no equivalent in the CLI; see the [API reference](https://franken.readthedocs.io/) for details.

### Step 1: cheap search (fixed length scale)

The GNN backbone is already configured above. Here we build the dataset configuration: we reuse the `train_path`/`val_path` loaded earlier, and use only 8 training samples to keep the computation light. Increasing `max_train_samples` can improve accuracy, but makes **every** `autotune` run below more expensive, since the feature map has to be computed for more structures.

In [ ]:
dataset_cfg = DatasetConfig(
    train_path=str(train_path),
    val_path=str(val_path),
    max_train_samples=8,
)

We fix the kernel's `length_scale` to a single, reasonable value (`10.0`) instead of searching over it — this is what makes this first `autotune` run cheap.

In [ ]:
rf_config = GaussianRFConfig(
    num_random_features=512,
    length_scale=10.0,
    rng_seed=42,
)

Solver hyperparameters are inexpensive to search because the costly feature maps can be reused. We test six logarithmically spaced L2 penalties from $10^{-10}$ to $10^{-5}$ and seven force weights from $10^{-3}$ to $10^3$.

The energy weight defaults to 1.0, and target weights are normalized internally. The configured force weight can therefore be read as the force-to-energy weight ratio.

In [ ]:
solver_cfg = SolverConfig(
    l2_penalty=HPSearchConfig(start=-10, stop=-5, num=6, scale="log"),
    force_weight=HPSearchConfig(start=-3, stop=3, num=7, scale="log"),
)

Group the configurations for this first, cheap search and run `autotune`. `run_dir` is the parent directory in which a unique run folder containing logs and checkpoints will be created.

Note: if you get **RuntimeError: CUDA out of memory** try to lower the `jac_chunk_size` to 8, 16 ,32, 64, etc. instead of "auto".

In [ ]:
autotune_cfg_step1 = AutotuneConfig(
    dataset=dataset_cfg,
    solver=solver_cfg,
    backbone=gnn_config,
    rfs=rf_config,
    metrics=["energy_MAE", "forces_MAE"],
    seed=42,
    jac_chunk_size="auto",
    run_dir="./results",
)

run_path_step1 = autotune(autotune_cfg_step1)

### Step 2: full search (also tune the length scale)

Now we broaden the kernel configuration to also search over `length_scale`, reusing the same `dataset_cfg`, `gnn_config`, and `solver_cfg` as before. This second run is more expensive (again, more so if you increase `max_train_samples`), since a separate feature map and model have to be fit for every length scale.

In [ ]:
rf_config = GaussianRFConfig(
    num_random_features=512,
    length_scale=HPSearchConfig(values=[1.0, 5.0, 10.0, 20.0, 30.0]),
    rng_seed=42,
)

In [ ]:
autotune_cfg_step2 = AutotuneConfig(
    dataset=dataset_cfg,
    solver=solver_cfg,
    backbone=gnn_config,
    rfs=rf_config,
    metrics=["energy_MAE", "forces_MAE"],
    seed=42,
    jac_chunk_size="auto",
    run_dir="./results",
)

run_path_step2 = autotune(autotune_cfg_step2)

## Analysing the results

We analyze here the results of the **full search** (Step 2, `run_path_step2`). The two main outputs are the model trained with the selected hyperparameters, saved as `best_ckpt.pt`, and JSON logs describing all trained models. The molecular-dynamics notebook shows how to use a trained model with ASE.

In [ ]:
# Load the logs for all trials and the selected best model.
with open(run_path_step2 / "log.json", "r") as fh:
    all_logs = json.load(fh)
with open(run_path_step2 / "best.json", "r") as fh:
    best_log = json.load(fh)

In [ ]:
best_ls = best_log["hyperparameters"]["random_features"]["length_scale"]
best_l2 = best_log["hyperparameters"]["solver"]["l2_penalty"]
best_fw = best_log["hyperparameters"]["solver"]["forces_weight"]
print("Best hyperparameters:")
print(f"	Length scale: {best_ls:.1f}")
print(f"	L2 penalty: {best_l2:.2e}")
print(f"	Force-to-energy weight ratio: {best_fw:.3g}")

To make the analysis easier, convert the JSON logs to a pandas DataFrame.

In [ ]:
logs_df = pd.json_normalize(all_logs)
logs_df

With three hyperparameters, it is difficult to visualize their effects simultaneously. First fix the length scale and L2 penalty at their selected values, then plot both validation errors as the force weight changes.

In [ ]:
force_weight_col = "hyperparameters.solver.forces_weight"
df_fw = logs_df[
    np.isclose(logs_df["hyperparameters.random_features.length_scale"], best_ls, atol=1e-10)
    & np.isclose(logs_df["hyperparameters.solver.l2_penalty"], best_l2, atol=1e-10)
].sort_values(force_weight_col)

fig, ax = plt.subplots()
energy_line = ax.plot(
    df_fw[force_weight_col],
    df_fw["metrics.validation.energy_MAE"],
    label="Energy MAE",
)
ax2 = ax.twinx()
forces_line = ax2.plot(
    df_fw[force_weight_col],
    df_fw["metrics.validation.forces_MAE"],
    label="Forces MAE",
    color="tab:red",
)
ax.set_xscale("log")
ax.legend(energy_line + forces_line, [line.get_label() for line in energy_line + forces_line])
ax.set_xlabel("Force-to-energy weight ratio")
ax.set_ylabel("Energy MAE [meV/atom]")
ax2.set_ylabel("Forces MAE [meV/Å]")
plt.show()

Next, fix equal raw energy and force weights (a ratio of 1) and inspect the effect of length scale and L2 penalty.

In [ ]:
df_kernel = logs_df[np.isclose(logs_df[force_weight_col], 1.0)]
pivot = df_kernel.pivot_table(
    index="hyperparameters.solver.l2_penalty",
    columns="hyperparameters.random_features.length_scale",
    values="metrics.validation.forces_MAE",
)

fig, ax = plt.subplots()
im = ax.imshow(pivot, vmax=pivot.values.min()*1.5, cmap="viridis_r", aspect="auto")
colorbar = fig.colorbar(im)
colorbar.set_label("Forces MAE [meV/Å]")
ax.set_xticks(range(len(pivot.columns)), pivot.columns)
ax.set_xlabel("Length scale")
ax.set_yticks(range(len(pivot.index)), [f"{value:.1e}" for value in pivot.index])
ax.set_ylabel("L2 penalty")